In [1]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

processor = WhisperProcessor.from_pretrained("openai/whisper-small")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

In [2]:
import soundfile as sf
import librosa

# make sure you have the same sampling rate as the one used during training (16000 for Whisper)
recording, sample_rate = sf.read("recording.wav")
recording = librosa.resample(recording, orig_sr=sample_rate, target_sr=16000)

In [3]:
recording

array([-4.86784211e-06, -7.51375319e-06, -4.40727308e-06, ...,
       -3.01258499e-03, -1.66545447e-03, -9.72301606e-03], shape=(80000,))

In [4]:
# preprocess the audio to extract features
inputs = processor.feature_extractor(
    recording, return_tensors="pt", sampling_rate=16000
).input_features
inputs

tensor([[[-0.7859, -0.7859, -0.7859,  ..., -0.7859, -0.7859, -0.7859],
         [-0.7859, -0.7859, -0.7859,  ..., -0.7859, -0.7859, -0.7859],
         [-0.7859, -0.7859, -0.7859,  ..., -0.7859, -0.7859, -0.7859],
         ...,
         [-0.7859, -0.7859, -0.7859,  ..., -0.7859, -0.7859, -0.7859],
         [-0.7859, -0.7859, -0.7859,  ..., -0.7859, -0.7859, -0.7859],
         [-0.7859, -0.7859, -0.7859,  ..., -0.7859, -0.7859, -0.7859]]])

In [5]:
# get the predicted token ids
predicted_ids = model.generate(inputs)
predicted_ids

Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.log

tensor([[2425, 7751,  393,  291, 1568,  452, 3177,  286, 1454,  291, 1074,  393,
         1568,  452, 3177,  490,  341]])

In [6]:
# decode token ids to text
transcribe = processor.tokenizer.batch_decode(predicted_ids, skip_special_tokens=True)[
    0
]
transcribe

' Hello hello can you hear my voice I hope you guys can hear my voice from this'